<a id="introduction"></a>

# Problem Set 5: Autoencoders
## CHEME 5820 · Machine Learning and Artificial Intelligence for Engineers · Spring 2025

**Name**: _______________________   **NetID**: _______________   **Date**: _______________

---

Handwritten digit images from the [MNIST database](https://en.wikipedia.org/wiki/MNIST_database) contain rich spatial structure — stroke patterns, curves, and loops that vary across writing styles yet remain recognizable as the same character. In this problem set, we build and train a **deterministic Autoencoder (AE)** that learns to compress each 784-pixel image into a low-dimensional **bottleneck code** and then reconstruct it from that code alone.

The autoencoder is a direct neural-network analogue of the embedding models you built in the previous problem set. In CBOW and Skip-Gram, a weight matrix $\mathbf{W}_1$ maps a sparse one-hot word vector into a dense embedding, and $\mathbf{W}_2$ decodes that embedding back into a prediction. An autoencoder does the same thing for images — but with deeper, non-linear encoder and decoder networks, and with reconstruction error as the training signal instead of cross-entropy.

> __Learning Objectives__
>
> By the end of this problem set, you should be able to:
> * __Implement an encoder and decoder using Flux.jl:__ Write `encode` and `decode` functions that map inputs to a low-dimensional bottleneck and back, using `Chain` and `Dense` layers with ReLU and sigmoid activations.
> * __Implement and minimise a reconstruction loss:__ Compute mean-squared error between the input and its reconstruction, and write a Flux.jl training loop that minimises it using the Adam optimiser.
> * __Visualise and interpret the learned latent space:__ Encode images to their bottleneck codes, decode them back, and linearly interpolate between two codes to test whether the AE has learned a smooth latent geometry.

Let's get started!

## Setup, Data, and Prerequisites
We set up the computational environment by including the `Include.jl` file, loading the
MNIST dataset, and setting up the required constants.

> __Environment Setup with Include.jl__
>
> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of `Include.jl` in the notebook's global scope. `Include.jl` sets paths, loads required external packages, and includes `src/Types.jl` and `src/Compute.jl`, which define the `MyAEModel` type and the helper functions used throughout this notebook.

In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

In addition to standard Julia libraries, we use [Flux.jl](https://fluxml.ai/Flux.jl/stable/) for automatic differentiation and neural network layers, [MLDatasets.jl](https://juliaml.github.io/MLDatasets.jl/stable/) to load MNIST, and [Plots.jl](https://docs.juliaplots.org/stable/) for visualization. `Include.jl` loads these packages and includes `src/Types.jl` (defines `MyAEModel`) and `src/Compute.jl` (defines the helper functions below).

> __Helper functions (provided in `src/Compute.jl`)__
>
> * `build_ae_model(input_dim, hidden_dim, latent_dim)`: Constructs a `MyAEModel` with a symmetric encoder–decoder architecture. The encoder maps $D \to H \to H/2 \to L$ using ReLU activations with no output activation at the bottleneck; the decoder inverts this path and ends with a sigmoid to constrain reconstructions to $[0, 1]$.
> * `load_mnist_digit(digit; n_examples)`: Loads MNIST training images for one digit class and returns a $(784 \times N)$ Float32 matrix with pixel values in $[0, 1]$.
> * `show_image_grid(X; nrows, ncols)`: Displays columns of a $784 \times N$ matrix as a grid of 28×28 greyscale images.

The three student-implemented functions — `encode`, `decode`, and `reconstruction_loss` — are the core of the autoencoder forward pass. Let's define them in the Implementations section below.

### Implementations

Complete the three functions below — they are the core of the autoencoder forward pass. All student-written code lives here so it is easy to find and edit in one place.

> __`encode(model, x)`__
>
> Maps a data batch $\mathbf{x} \in \mathbb{R}^{D \times N}$ to bottleneck codes $\mathbf{z} \in \mathbb{R}^{L \times N}$ by passing `x` through `model.encoder`. The encoder is a three-layer `Chain` ($784 \to 256 \to 128 \to L$) with ReLU activations and no output activation at the bottleneck, leaving the codes unconstrained.

> __`decode(model, z)`__
>
> Maps bottleneck codes $\mathbf{z} \in \mathbb{R}^{L \times N}$ back to pixel space $\hat{\mathbf{x}} \in \mathbb{R}^{D \times N}$ by passing `z` through `model.decoder`. The decoder is a three-layer `Chain` ($L \to 128 \to 256 \to 784$) ending with a sigmoid, which constrains reconstructions to $[0, 1]$ to match the normalised pixel range.

> __`reconstruction_loss(model, x)`__
>
> Computes the mean-squared reconstruction error: encode `x` to `z`, decode `z` to `x̂`, and return `mean(sum((x .- x̂).^2; dims=1))`. The inner `sum` accumulates squared pixel errors over the $D = 784$ dimensions for each example independently; the outer `mean` averages across the $N$ examples in the batch.

Let's implement the three functions.

In [ ]:
# Problem 2a — encode  (15 pts)
# --------------------------------------------------------------------------
# encode maps a batch of inputs x (D × N) to bottleneck codes z (L × N).
#
# Steps:
#   1. Pass x through model.encoder
#   2. Return the result

function encode(model::MyAEModel, x::AbstractMatrix)
    # YOUR CODE HERE

end

# Problem 2b — decode  (15 pts)
# --------------------------------------------------------------------------
# decode maps bottleneck codes z (L × N) to reconstructed inputs x̂ (D × N).
#
# Steps:
#   1. Pass z through model.decoder
#   2. Return the result
#      (the final sigmoid layer already constrains output to [0, 1])

function decode(model::MyAEModel, z::AbstractMatrix)
    # YOUR CODE HERE

end

# Problem 3a — reconstruction_loss  (15 pts)
# --------------------------------------------------------------------------
# reconstruction_loss should:
#   1. Encode x  →  z   using encode(model, x)
#   2. Decode z  →  x̂  using decode(model, z)
#   3. Return the MSE:  mean(sum((x .- x̂).^2; dims=1))

function reconstruction_loss(model::MyAEModel, x::AbstractMatrix)
    # YOUR CODE HERE

end

With the three functions defined, let's set the constants that control the dataset, model architecture, and training schedule. The comment next to each constant describes its purpose and permissible values.

In [ ]:
DIGIT         = 3;       # MNIST digit class to model (0–9)
K             = 100;     # number of training examples
D             = 784;     # input dimension: 28 × 28 = 784 pixels
L             = 8;       # bottleneck (latent) dimension
H             = 256;     # hidden-layer width
LR            = 1f-3;    # Adam learning rate
NUM_EPOCHS    = 2_000;   # training epochs

Random.seed!(42);

___
### Background: Autoencoders

An autoencoder is a neural network trained to reproduce its own input at the output layer,
subject to passing through a low-dimensional **bottleneck**. It consists of two sub-networks:

| Component | Maps | Role |
|-----------|------|------|
| **Encoder** $f_\theta$ | $\mathbf{x}\in\mathbb{R}^D \to \mathbf{z}\in\mathbb{R}^L,\; L\ll D$ | compresses input to a compact code |
| **Decoder** $g_\phi$ | $\mathbf{z}\in\mathbb{R}^L \to \hat{\mathbf{x}}\in\mathbb{R}^D$ | reconstructs the input from the code |

Both are trained jointly to minimise the **reconstruction loss** over the training set:

$$\mathcal{L}(\theta,\phi) = \frac{1}{N}\sum_{i=1}^{N}\|\mathbf{x}_i - g_\phi(f_\theta(\mathbf{x}_i))\|^2$$

Because the only path from input to output passes through the $L$-dimensional bottleneck,
the encoder is forced to retain only the most important structure of the data — and the
decoder must learn to recover the full input from that compressed representation.

> __Connection to CBOW and Skip-Gram__
>
> You have already built models with this compress-then-reconstruct structure. In CBOW,
> the input weight matrix $\mathbf{W}_1$ acts as an encoder that maps a sparse one-hot
> word vector into a dense $d_h$-dimensional embedding, and $\mathbf{W}_2$ decodes that
> embedding back into a prediction over the vocabulary. An autoencoder is the same idea
> applied to continuous inputs (images) with deeper, non-linear encoder and decoder networks.

### Data Loading and Exploration

We train the autoencoder on $K = 100$ examples of MNIST digit **3** from the training split. Each 28×28 greyscale image is flattened to a 784-dimensional vector and stored as a column of the data matrix $\mathbf{X} \in \mathbb{R}^{784 \times K}$, with pixel values in $[0, 1]$.

> __`load_mnist_digit(DIGIT; n_examples = K)`__
>
> Queries `MLDatasets.MNIST(:train)` for all images of digit `DIGIT`, selects the first `K`, flattens each 28×28 array to a 784-element column vector, and returns a `(784 × K)` Float32 matrix. Pixel values are already normalised to $[0, 1]$ by MLDatasets.

Let's load the data **(1a)**, compute pixel statistics **(1b)**, and visualise a sample grid **(1c)**.

In [ ]:
# Problem 1a — load the data  (5 pts)
# --------------------------------------------------------------------------
# YOUR CODE HERE: call load_mnist_digit, store the result in X

X = missing;

println("Data matrix size: ", size(X))   # expected: (784, 100)

> __Pixel statistics__
>
> With the data loaded, let's characterise the pixel value distribution of the training set. The mean pixel value of MNIST digit images is well below 0.5 — most pixels are dark background — which explains why the initial reconstruction loss is large: the decoder must learn to reproduce many near-zero pixels.

Let's compute the mean and standard deviation of all pixel values in `X` and store the results in `μ_data::Float32` and `σ_data::Float32`.

In [ ]:
# Problem 1b — pixel statistics  (5 pts)
# --------------------------------------------------------------------------
# YOUR CODE HERE: compute mean and std of all pixel values in X

μ_data = missing;
σ_data = missing;

@printf("Mean pixel value: %.4f\n", μ_data);
@printf("Std  pixel value: %.4f\n", σ_data);

In [ ]:
# ── Autograder: Problem 1 ────────────────────────────────────────────────────
check!(GRADER, "P1a", "X loaded, shape (784, 100)",   5, () -> !ismissing(X) && size(X) == (D, K))
check!(GRADER, "P1b", "μ_data ≈ mean pixel value",    3, () -> !ismissing(μ_data) && μ_data ≈ mean(X))
check!(GRADER, "P1b", "σ_data ≈ std pixel value",     2, () -> !ismissing(σ_data) && σ_data ≈ std(X))

> __Visualising the training data__
>
> Before building the model, let's look at a sample of the digit images we are working with. `show_image_grid` reshapes each 784-element column of `X` back to a 28×28 array and renders it as a greyscale heatmap, so we can verify the data loaded correctly and get an intuition for the variation in handwriting style across examples.

Let's display a 4×4 grid of training examples.

In [ ]:
# Problem 1c — visualise training examples  (5 pts)
show_image_grid(X; nrows=4, ncols=4)

___
## Problem 1 — Autoencoder Architecture  (30 points)

The autoencoder forward pass requires two operations: the **encoder** compresses each $D = 784$-pixel input to an $L = 8$-dimensional bottleneck code, and the **decoder** reconstructs the input from that code alone. Both are one-liners that call through the `Chain` networks stored in `model.encoder` and `model.decoder` — scroll up to the **Implementations** section to complete `encode` and `decode`.

> __`build_ae_model(D, H, L)`__
>
> Constructs a `MyAEModel` with a symmetric architecture. The encoder compresses $784 \to 256 \to 128 \to L = 8$ using ReLU activations, with **no activation on the bottleneck** so that codes are unconstrained real numbers. The decoder inverts this path ($8 \to 128 \to 256 \to 784$) and ends with a **sigmoid** that constrains reconstructions to $[0, 1]$, matching the normalised pixel range.

Let's build the model and inspect its architecture.

In [ ]:
# Build the autoencoder — nothing to change here
ae = build_ae_model(D, H, L);
println("AE created.")
println("  Encoder : ", ae.encoder)
println("  Decoder : ", ae.decoder)

Let's run the autograder to verify that `encode` and `decode` produce the correct shapes and value ranges before moving on to training.

In [ ]:
# ── Autograder: Problem 2 ────────────────────────────────────────────────────
let
    x_test = X[:, 1:5]
    z_test = encode(ae, x_test)
    x̂_test = decode(ae, z_test)

    check!(GRADER, "P2a", "encode: output shape is (L, N)",   8, () -> size(z_test) == (L, 5))
    check!(GRADER, "P2a", "encode: output has no NaN",        7, () -> !any(isnan, z_test))
    check!(GRADER, "P2b", "decode: output shape is (D, N)",   8, () -> size(x̂_test) == (D, 5))
    check!(GRADER, "P2b", "decode: output in [0, 1]",         7, () -> all(0f0 .≤ x̂_test .≤ 1f0))
end

___
## Problem 2 — Reconstruction Loss and Training  (30 points)

With the encoder and decoder in place, we now define the training objective and run the optimisation loop. The **reconstruction loss** measures how faithfully the autoencoder recovers its inputs after passing them through the bottleneck. Training minimises this loss jointly over all encoder and decoder parameters using the Adam optimiser.

Scroll up to the **Implementations** section and complete `reconstruction_loss` **(2a)**. It should implement:

$$\mathcal{L}(\theta,\phi;\,\mathbf{X}) = \frac{1}{N}\sum_{i=1}^{N}\|\mathbf{x}_i - \hat{\mathbf{x}}_i\|^2 = \texttt{mean}(\texttt{sum}((\mathbf{X} - \hat{\mathbf{X}})^{\odot 2};\,\text{dims}=1))$$

**Hint:** `sum(...; dims=1)` sums the $D = 784$ squared pixel errors for each example independently (giving a $1 \times N$ row vector); `mean(...)` then averages across the $N$ examples in the batch.

Let's verify `reconstruction_loss` with the autograder before running the training loop.

In [ ]:
# ── Autograder: Problem 3a ───────────────────────────────────────────────────
let
    x_test = X[:, 1:5]
    loss   = reconstruction_loss(ae, x_test)

    check!(GRADER, "P3a", "loss is a scalar Float",   5, () -> loss isa AbstractFloat)
    check!(GRADER, "P3a", "loss is positive",          5, () -> loss > 0)
    check!(GRADER, "P3a", "loss < D (sanity bound)",   5, () -> loss < D)
end

 > __Training with `Flux.withgradient` and `Flux.update!`__ **(2b)**
>
> The training loop calls `Flux.withgradient(ae) do m ... end` on each epoch. Inside the `do` block, `m` is a differentiable handle to `ae` — call `reconstruction_loss(m, X)` there so Flux can trace gradients through the full encode–decode chain. `Flux.update!` then applies the Adam step to all encoder and decoder parameters at once.

Let's train the autoencoder for `NUM_EPOCHS = 2000` epochs and store the per-epoch loss in `losses`.

In [ ]:
# Problem 3b — complete the training loop  (15 pts)
# --------------------------------------------------------------------------
opt_state = Flux.setup(Adam(LR), ae);
losses    = Float32[];

println("Training autoencoder...");
for epoch in 1:NUM_EPOCHS
    loss, grads = Flux.withgradient(ae) do m
        # YOUR CODE HERE — one line: call reconstruction_loss on the full dataset X
        
    end;
    Flux.update!(opt_state, ae, grads[1]);
    push!(losses, loss);
    epoch % 400 == 0 && @printf("  Epoch %4d | loss = %.4f\n", epoch, loss);
end
println("Training complete.");

In [ ]:
# ── Autograder: Problem 3b ───────────────────────────────────────────────────
check!(GRADER, "P3b", "losses has NUM_EPOCHS entries",   5, () -> length(losses) == NUM_EPOCHS)
check!(GRADER, "P3b", "final loss < initial loss",       7, () -> losses[end] < losses[1])
check!(GRADER, "P3b", "loss reduced by ≥ 50%",          3, () -> losses[end] < 0.5f0 * losses[1])

Let's plot the per-epoch training loss to inspect how the autoencoder converged.

In [ ]:
plot(losses;
    xlabel="Epoch", ylabel="Reconstruction Loss (MSE)",
    title="Autoencoder Training", label="MSE",
    color=:steelblue, lw=2, framestyle=:box)

 > __What do we observe?__
>
> The loss curve should decrease steeply at first and then flatten as the encoder and decoder settle into a stable compressed representation. It will not reach zero — some reconstruction error is irreducible given the $L = 8$-dimensional bottleneck — but a well-trained model should reduce the loss to well below its initial value.

___
## Problem 3 — Latent Space Analysis  (25 points)

After training, we evaluate what the autoencoder has learned. First, we compare original images to their reconstructions **(3a)** to gauge how much detail the $L = 8$-dimensional bottleneck retains. Then, we test whether the bottleneck is **smooth** **(3b)** by linearly interpolating between two training images in latent space — if the decoded path transitions gradually from one image to the other, the encoder has learned a geometrically meaningful representation.

 > __Reconstruction: originals vs. decoded **(3a)**__
>
> We encode eight training images to their $L = 8$-dimensional bottleneck codes and decode back to pixel space. The left panel shows the originals; the right panel shows the reconstructions. Well-trained reconstructions should be recognisable as the correct digit and capture the overall stroke structure, though some blurring is expected given the 98× compression.

Let's compare the originals and their reconstructions.

In [ ]:
# Problem 4a — reconstruction quality  (10 pts)
let
    n_show  = 8;
    x_orig  = X[:, 1:n_show];
    z_orig  = encode(ae, x_orig);
    x_recon = decode(ae, z_orig);

    p_orig  = show_image_grid(x_orig;  nrows=2, ncols=4);
    p_recon = show_image_grid(x_recon; nrows=2, ncols=4);
    plot(p_orig, p_recon; layout=(1, 2), size=(700, 250),
         plot_title="Left: originals   Right: reconstructions")
end

 > __Latent-space interpolation **(3b)**__
>
> A smooth latent space means that moving along a straight line between two codes $\mathbf{z}_1$ and $\mathbf{z}_2$ produces a coherent sequence of decoded images. We test this by computing $\mathbf{z}_\alpha = (1 - \alpha)\mathbf{z}_1 + \alpha\mathbf{z}_2$ for ten evenly-spaced values of $\alpha \in [0, 1]$ and decoding the full path in a single batched call.

Let's encode two images, interpolate between their codes, and display the decoded path.

In [ ]:
# Problem 4b — latent-space interpolation  (15 pts)
# --------------------------------------------------------------------------
n_steps = 10;
x1 = X[:, 1:1];   # first training image   (784 × 1)
x2 = X[:, 6:6];   # sixth training image   (784 × 1)

# YOUR CODE HERE
# 1. z1 = encode(ae, x1),  z2 = encode(ae, x2)
# 2. For each α in range(0f0, 1f0, length=n_steps): z_α = (1-α)*z1 + α*z2
# 3. Stack into z_path  (L × n_steps),  then x_path = decode(ae, z_path)
# 4. Call show_image_grid(x_path; nrows=2, ncols=5)

z1 = missing;
z2 = missing;

alphas = range(0f0, 1f0; length=n_steps);
z_path = missing;   # L × n_steps matrix of interpolated codes
x_path = missing;   # decode all at once

show_image_grid(x_path; nrows=2, ncols=5)

In [ ]:
# ── Autograder: Problem 4b ───────────────────────────────────────────────────
check!(GRADER, "P4b", "z1 has shape (L, 1)",            2, () -> !ismissing(z1) && size(z1) == (L, 1))
check!(GRADER, "P4b", "z2 has shape (L, 1)",            2, () -> !ismissing(z2) && size(z2) == (L, 1))
check!(GRADER, "P4b", "z_path has shape (L, n_steps)",  3, () -> !ismissing(z_path) && size(z_path) == (L, n_steps))
check!(GRADER, "P4b", "x_path has shape (D, n_steps)",  3, () -> !ismissing(x_path) && size(x_path) == (D, n_steps))

___
<a id="discussion"></a>

## Discussion
Use the results from Problems 1–3 to answer the discussion questions below. Edit the `> __Answer__` block under each question with your response (2–4 sentences).

---

**DQ1: Reconstruction quality and bottleneck capacity.** The autoencoder compresses each $D = 784$-pixel image into an $L = 8$-dimensional bottleneck code — a 98× reduction. The encoder must decide what structure to keep and what to discard.

> __Strategy__: Examine the reconstruction panel from Problem 3. Are the reconstructions sharp or blurry? Change `L` in the constants block to `2`, `4`, `16`, and `32`, re-run all cells, and compare reconstruction quality across these settings. What fine-grained information appears to be discarded first as the bottleneck shrinks?

> __Answer__: *Fill in your answer here.*

---

**DQ2: Latent space smoothness.** Linear interpolation between two bottleneck codes $\mathbf{z}_1$ and $\mathbf{z}_2$ tests whether the autoencoder has learned a smooth, well-structured latent space — or whether nearby codes decode to very different images.

> __Strategy__: Examine the interpolation grid from Problem 3. Does the sequence transition smoothly from one image to another, or do intermediate frames look incoherent? Try changing `L` and re-running the interpolation cell. What does a smooth (or fragmented) transition reveal about the geometry of the learned latent space?

> __Answer__: *Fill in your answer here.*

---

**DQ3: Generative limits of the standard autoencoder.** A standard AE places no explicit constraint on the distribution of bottleneck codes — the encoder is free to use any region of $\mathbb{R}^L$ that minimises reconstruction loss.

> __Strategy__: Run `extrema(encode(ae, X))` to inspect the range of the learned codes. Then sample `z_rand = randn(Float32, L, 1)` and decode it with `show_image_grid(decode(ae, z_rand))`. Does the output look like a realistic digit? Explain why or why not, and describe what additional constraint would be needed to make the AE a proper generative model.

> __Answer__: *Fill in your answer here.*

In [ ]:
# Set each flag to true after you have written your answer above
did_I_answer_DQ1 = false;   # change to true after answering DQ1
did_I_answer_DQ2 = false;   # change to true after answering DQ2
did_I_answer_DQ3 = false;   # change to true after answering DQ3

In [ ]:
# ── Autograder: Discussion ────────────────────────────────────────────────────
check!(GRADER, "Discussion", "answered DQ1",   4, () -> did_I_answer_DQ1 == true)
check!(GRADER, "Discussion", "answered DQ2",   3, () -> did_I_answer_DQ2 == true)
check!(GRADER, "Discussion", "answered DQ3",   3, () -> did_I_answer_DQ3 == true)

In [ ]:
# ── Final Score ───────────────────────────────────────────────────────────────
score!(GRADER)